# N-real · 真实 LLM 评委 (TinyLlama) + 位置偏置

> **小而真** (配套 llm-judge-arena) · 用 LLM 当评委 (LLM-as-judge) 是当下主流自动评估。
> 这里用**真实 TinyLlama** 当评委判两个答案谁更好, 并亲手暴露一个真实病灶: **位置偏置** (换个顺序结论就变)。
> CPU 离线确定性。

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1] / "_shared"))
import realmodels as rm
import numpy as np, torch
print("真实模型可用性:", rm.available())

C:\Users\ericp\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


真实模型可用性: {'gpt2': True, 'TinyLlama/TinyLlama-1.1B-Chat-v1.0': True}


> 若上面显示某模型为 False, 表示本机无该 HF 缓存, 真实例子会自动跳过 (不影响本专题的 toy notebook)。

## 1. 让真实模型当评委

In [2]:
tok, model = rm.tinyllama()
question = "What is the capital of Japan?"
ans_good = "The capital of Japan is Tokyo."
ans_bad  = "The capital of Japan is Kyoto, which has always been the capital."

def judge(a, b):
    prompt = (f"Question: {question}\n\nAnswer A: {a}\nAnswer B: {b}\n\n"
              "Which answer is more accurate? Reply with only 'A' or 'B'.")
    return rm.chat(tok, model, prompt, max_new_tokens=6).strip()

if model is not None:
    v1 = judge(ans_good, ans_bad)   # 好答案在 A 位
    v2 = judge(ans_bad, ans_good)   # 好答案在 B 位 (交换)
    print(f"好答案放 A 位 → 评委选: {v1!r}  (正确应选 A)")
    print(f"好答案放 B 位 → 评委选: {v2!r}  (正确应选 B)")
else:
    print("无 TinyLlama, 跳过"); v1=v2=None

好答案放 A 位 → 评委选: 'Both answers are correct.'  (正确应选 A)
好答案放 B 位 → 评委选: 'Both answers are correct.'  (正确应选 B)


## 2. 位置偏置: 同样两个答案, 顺序一换结论可能就变

In [3]:
if v1 is not None:
    picks_good_1 = (v1 == 'A')   # A 位是好答案
    picks_good_2 = (v2 == 'B')   # B 位是好答案
    consistent = picks_good_1 and picks_good_2
    print(f"两次都选中好答案? {consistent}")
    if not consistent:
        print("→ 出现**位置偏置**: 评委的选择受答案出现位置影响, 不只看内容。")
    else:
        print("→ 这次一致; 但位置偏置是 LLM 评委的已知普遍病灶, 大规模评测必现。")
    print('''
缓解办法 (你在 judge 模块学的):
  - 两个顺序都判, 取一致结果 (本 cell 做的就是双向检查)。
  - 多评委投票 / 自洽性。
  - 校准 prompt、给评分标准 (rubric)。''')

两次都选中好答案? False
→ 出现**位置偏置**: 评委的选择受答案出现位置影响, 不只看内容。

缓解办法 (你在 judge 模块学的):
  - 两个顺序都判, 取一致结果 (本 cell 做的就是双向检查)。
  - 多评委投票 / 自洽性。
  - 校准 prompt、给评分标准 (rubric)。


## 3. 反思
- 你用**真实模型**当了评委, 也亲手暴露了**位置偏置**: LLM 评委不是中立尺子, 有系统偏差。
- 其它已知偏置: 偏长答案、偏自己风格、偏有礼貌措辞 —— 都需校准。
- 所以 LLM-judge 要工程化: 双向/多评委/rubric/和人评对齐 (你 judge 模块的核心)。

> 教训: 自动评估强大但**有偏**; 用它前必须测它自己的偏差 (元评估)。